# FIVES — cross-dataset evaluation, qualitative figures, in-domain training

| `MODE` | What it does | Inputs to attach (besides FIVES) | GPU time |
|---|---|---|---|
| `crosseval` | RFMiD checkpoints tested on the FIVES test set (FOV + vessel metrics). Works with any subset of checkpoints (e.g. only `lorentz_seed0`). | RFMiD checkpoints (`results/<model>_seed<k>/best.pt`) | ≈ 5 min / checkpoint |
| `figure` | Qualitative figures (heavy noise): FIVES with vessel annotation + RFMiD. Needs E, P and L for `FIG_SEED`. | RFMiD checkpoints + `retinal-disease-classification` | ≈ 5 min |
| `train` | Train + evaluate `fives_<EXPERIMENT>` for `SEEDS` on FIVES. | – | E 15 min, L 1 h, P 1.6 h / seed |

**Setup:** GPU T4 · Internet ON · Add Input → `nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation` (+ the inputs above).
Then **Save Version → Save & Run All (Commit)** and download the zip from the Output tab.

In [ ]:
MODE = "crosseval"              # crosseval | figure | train
EXPERIMENT = "lorentz"         # train mode: euclidean | poincare | lorentz
SEEDS = [0, 1, 2]              # train mode
FIG_SEED = 1                   # figure mode: seed whose E/P/L checkpoints are shown
REPO_URL = "https://github.com/hiayoub/poincare-vs-lorentz-denoising.git"
BRANCH = "main"

In [ ]:
from pathlib import Path
EXTS = (".png", ".jpg", ".jpeg")
fives = sorted(Path("/kaggle/input").glob("**/train/Original"))
assert fives, "FIVES NOT ATTACHED: Add Input -> nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation"
FIVES_ROOT = fives[0].parent.parent
n = {s: sum(f.suffix.lower() in EXTS for f in (FIVES_ROOT / s / "Original").iterdir()) for s in ("train", "test")}
print("FIVES at", FIVES_ROOT, "images:", n)   # the Kaggle copy has 1 extra image in train/ (ignored by the split)
assert n["test"] == 200 and n["train"] >= 600, f"unexpected FIVES size {n}"
ckpts = sorted(Path("/kaggle/input").glob("**/results/*_seed*/best.pt"))
print(len(ckpts), "checkpoints:", sorted({c.parent.name for c in ckpts}))
if MODE in ("crosseval", "figure"):
    assert ckpts, "No checkpoints: attach the RFMiD training outputs / checkpoint dataset"
if MODE == "figure":
    assert sorted(Path("/kaggle/input").glob("**/Training_Set")), "RFMiD NOT ATTACHED (needed for the RFMiD figure)"
    have = {c.parent.name for c in ckpts}
    missing = [f"{m}_seed{FIG_SEED}" for m in ("euclidean", "poincare", "lorentz") if f"{m}_seed{FIG_SEED}" not in have]
    assert not missing, f"figure needs {missing}"

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
print(subprocess.check_output(["git", "log", "-1", "--format=commit %h  %s"]).decode())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "third_party/hypll", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips==0.1.4", "torchmetrics>=1.4", "pytest"])
Path("configs/eval_fives_local.yaml").write_text(
    Path("configs/eval_fives.yaml").read_text().replace("root: null", f"root: '{FIVES_ROOT}'"))

In [ ]:
!python -m pytest tests -q
if _exit_code != 0:
    raise RuntimeError("tests failed")

## Run

In [ ]:
import shutil
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

def copy_ckpts():
    seen = set()
    for ck in ckpts:
        name = ck.parent.name
        if name in seen: continue
        seen.add(name)
        shutil.copytree(ck.parent, OUT / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("last.pt"))
    return sorted(seen)

if MODE == "crosseval":
    for name in copy_ckpts():
        !python -m hypdenoise.evaluate --run {OUT / name} --data-config configs/eval_fives_local.yaml --tag fives
        if _exit_code != 0:
            raise RuntimeError(f"cross-eval failed for {name}")
    if any(p.name.startswith("euclidean_") for p in OUT.iterdir()) and len(list(OUT.glob("*_seed*"))) > 1:
        !python -m hypdenoise.compare --results {OUT} --reference euclidean --tag fives
elif MODE == "figure":
    copy_ckpts()
    # Images = 50th / 75th percentile of the per-image hyperbolic gain (heavy noise, seed-averaged),
    # computed from the cross-eval / FOV CSVs: FIVES on vessel PSNR, RFMiD on FOV PSNR. Not cherry-picked.
    !python scripts/qualitative_figure.py --results {OUT} --seed {FIG_SEED} --data-config configs/eval_fives_local.yaml --metric psnr_vessel --images 178_N.png 85_D.png --out {OUT}/figures/qualitative_fives_heavy
    if _exit_code != 0:
        raise RuntimeError("FIVES figure failed")
    !python scripts/qualitative_figure.py --results {OUT} --seed {FIG_SEED} --images 235.png 10.png --out {OUT}/figures/qualitative_rfmid_heavy
else:
    ROOT = f"data.root='{FIVES_ROOT}'"
    for seed in SEEDS:
        !python -m hypdenoise.train --config configs/fives_{EXPERIMENT}.yaml --seed {seed} --set output_dir={OUT} {ROOT}
        if _exit_code != 0:
            raise RuntimeError(f"training failed for seed {seed}")
        !python -m hypdenoise.evaluate --run {OUT}/fives_{EXPERIMENT}_seed{seed}
        if _exit_code != 0:
            raise RuntimeError(f"evaluation failed for seed {seed}")

## Package the results (Output tab → zip, without the .pt files)

In [ ]:
# Copied checkpoints are already in the inputs; trained ones stay in /kaggle/working/results (Output tab)
if MODE != "train":
    for p in OUT.glob("*/best.pt"): p.unlink()
os.chdir("/kaggle/working")
shutil.rmtree(REPO, ignore_errors=True)
small = Path("/kaggle/working/_small")
shutil.copytree(OUT, small, ignore=shutil.ignore_patterns("*.pt"), dirs_exist_ok=True)
shutil.make_archive(f"/kaggle/working/fives_{MODE}{'_' + EXPERIMENT if MODE == 'train' else ''}", "zip", small)
shutil.rmtree(small)
!ls -la /kaggle/working